In [ ]:
from wings.modeling.unet import UNet
from wings.config import MODELS_DIR
import torch

model = UNet(in_channels=1, out_channels=1, kernel_size=3)
model


In [ ]:
checkpoint = torch.load(MODELS_DIR / "unet-pytorch.pt")
state_dict = checkpoint

old_weight = state_dict["encoder1.enc1conv1.weight"]

new_weight = old_weight.mean(dim=1, keepdim=True)

print(old_weight.shape)
print(new_weight.shape)

state_dict["encoder1.enc1conv1.weight"] = new_weight


In [ ]:
model.load_state_dict(state_dict)

# From `lightning`

In [9]:
import torch


def inflate_3x3_to_5x5(w):
    w5 = torch.zeros(
        w.shape[0],
        w.shape[1],
        5,
        5,
        dtype=w.dtype,
        device=w.device,
    )
    w5[:, :, 1:4, 1:4] = w
    return w5


def load_lightning_3x3_into_unet_5x5(model_5x5, checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location="cpu")
    old_state = ckpt["state_dict"]

    new_model_state = model_5x5.state_dict()
    converted_state = {}

    loaded_direct = 0
    inflated = 0
    skipped = []

    for key, value in old_state.items():
        # Lightning checkpoint has "model." prefix
        if key.startswith("model."):
            key = key[len("model."):]

        if key not in new_model_state:
            skipped.append((key, "not in new model"))
            continue

        target = new_model_state[key]

        if value.shape == target.shape:
            converted_state[key] = value
            loaded_direct += 1

        elif (
            value.ndim == 4
            and target.ndim == 4
            and value.shape[:2] == target.shape[:2]
            and value.shape[-2:] == (3, 3)
            and target.shape[-2:] == (5, 5)
        ):
            converted_state[key] = inflate_3x3_to_5x5(value)
            inflated += 1

        else:
            skipped.append((key, f"{tuple(value.shape)} -> {tuple(target.shape)}"))

    missing, unexpected = model_5x5.load_state_dict(converted_state, strict=False)

    print(f"Loaded directly: {loaded_direct}")
    print(f"Inflated 3x3 -> 5x5: {inflated}")
    print(f"Skipped: {len(skipped)}")
    print(f"Missing: {len(missing)}")
    print(f"Unexpected: {len(unexpected)}")

    if skipped:
        print("\nSkipped examples:")
        for item in skipped[:20]:
            print(item)

    return model_5x5

In [11]:
from wings.modeling.loss import DiceLoss, WeightedDiceLoss
from wings.modeling.litnet import LitNet
from wings.config import PROCESSED_DATA_DIR, MODELS_DIR
from wings.dataset import MaskRectangleDataset
from wings.modeling.unet import UNet
import torch



checkpoint_path = MODELS_DIR / "new_unet" / "custom-unet-pretrained-epoch=49-val_loss=0.02-custom-unet-training_1.ckpt"


model_5x5 = UNet(
    in_channels=1,
    out_channels=1,
    init_features=32,
    kernel_size=5,
    sigmoid=True,
)

model_5x5 = load_lightning_3x3_into_unet_5x5(
    model_5x5,
    checkpoint_path,
)

# checkpoint = torch.load(checkpoint_path, map_location="cpu")
# checkpoint

Loaded directly: 100
Inflated 3x3 -> 5x5: 18
Skipped: 0
Missing: 0
Unexpected: 0
